# 01 · Lakehouse Foundations

This notebook introduces the Databricks lakehouse environment and builds a small Delta table from scratch.

### Skills demonstrated
- Apache Spark environment inspection
- Lakehouse architecture concepts
- Delta table creation and transaction history
- Catalog and schema exploration
- Basic Databricks filesystem operations

This notebook is a cleaned-up version of a hands-on learning lab. The goal is to show the concepts I practiced and the code I used to apply them.


## 1. Verify the Spark environment


In [ ]:
print(f"Apache Spark version: {spark.version}")


## 2. Compare data architectures

Comparison of common capabilities of warehouse, a data lake, and a lakehouse.


In [ ]:
architectures = spark.createDataFrame(
    [
        ("Data Warehouse", True, True, False, False, True),
        ("Data Lake", False, False, True, True, False),
        ("Data Lakehouse", True, True, True, True, True),
    ],
    [
        "architecture",
        "acid_transactions",
        "schema_enforcement",
        "unstructured_data",
        "low_cost_storage",
        "bi_support",
    ],
)

display(architectures)


## 3. Create a Delta table

Delta Lake adds transactional guarantees and table history on top of data stored in the lakehouse.


In [ ]:
# Keep portfolio objects grouped in their own schema.
spark.sql("CREATE SCHEMA IF NOT EXISTS portfolio_lab")

data = [
    (1, "John", 100.0),
    (2, "Jane", 200.0),
    (3, "Bob", 300.0),
    (4, "Alice", 400.0),
    (5, "Charlie", 500.0),
    (6, "Dave", 600.0),
]

sample_df = spark.createDataFrame(data, ["id", "name", "value"])

(
    sample_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("portfolio_lab.lakehouse_demo")
)

display(spark.table("portfolio_lab.lakehouse_demo"))


## 4. Inspect Delta transaction history

`DESCRIBE HISTORY` shows the operations recorded in the Delta transaction log

In [ ]:
%sql
DESCRIBE HISTORY portfolio_lab.lakehouse_demo


## 5. Explore catalogs and schemas


In [ ]:
%sql
SHOW CATALOGS


In [ ]:
%sql
SHOW SCHEMAS


## 6. Create and query a managed SQL table


In [ ]:
%sql
CREATE OR REPLACE TABLE portfolio_lab.cities (
    name STRING,
    state STRING,
    population INT
);

INSERT INTO portfolio_lab.cities VALUES
    ('Guadalajara', 'Jalisco', 1385629),
    ('León', 'Guanajuato', 1579803),
    ('Monterrey', 'Nuevo León', 1142994);

SELECT *
FROM portfolio_lab.cities
ORDER BY population DESC;


## 7. Explore Databricks sample files

`dbutils.fs.ls` provides access to the Databricks filesystem


In [ ]:
files = dbutils.fs.ls("/databricks-datasets/")
for file in files[:10]:
    print(file.path)


## Key takeaways

- A lakehouse combines flexible data-lake storage with warehouse-style management features.
- Delta tables provide transaction history in addition to tabular storage.
- Databricks exposes both Python and SQL interfaces over the same platform.
- Catalogs and schemas keep data assets organized as projects grow.
